# Paired benchmark uncertainty

This notebook computes paired 95% patch-cluster bootstrap intervals for the principal BEN.txt condition and coordinate-counterfactual contrasts. It resamples `patch_id`, retains all task rows belonging to each selected patch, and reports every difference as system A minus system B. These intervals describe benchmark-sampling uncertainty; training-seed variation is reported separately.

In [ ]:
from pathlib import Path
import sys

import pandas as pd
from IPython.display import display

repo_root = Path.cwd()
if not (repo_root / 'pyproject.toml').is_file():
    repo_root = repo_root.parent
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from notebooks.utils.paired_bootstrap import (
    DIRECT_GEOGRAPHY_METRICS,
    PRIMARY_METRICS,
    PredictionComparison,
    analyze_comparisons,
)

evaluation_root = repo_root / 'outputs' / 'evaluation'
analysis_dir = repo_root / 'notebooks' / 'analysis'


In [ ]:
comparison_specs = [
    ('2B seed 42: loc_text - no_loc', 'core', 'loc_text', '11441', 'no_loc', '11437'),
    ('2B seed 42: loc_embed - no_loc', 'core', 'loc_embed', '11438', 'no_loc', '11437'),
    ('2B seed 43: loc_text - no_loc', 'core', 'loc_text', '11624', 'no_loc', '11622'),
    ('2B seed 43: loc_embed - no_loc', 'core', 'loc_embed', '11627', 'no_loc', '11622'),
    ('4B seed 42: loc_text - no_loc', 'core', 'loc_text', '11443', 'no_loc', '11442'),
    ('4B seed 42: loc_embed - no_loc', 'core', 'loc_embed', '11444', 'no_loc', '11442'),
    ('8B seed 42: loc_text - no_loc', 'core', 'loc_text', '11682', 'no_loc', '11680'),
    ('8B seed 42: loc_embed - no_loc', 'core', 'loc_embed', '11685', 'no_loc', '11680'),
    ('2B seed 42: loc_text shuffled - correct', 'counterfactual', 'loc_text shuffled', '11445', 'loc_text correct', '11441'),
    ('2B seed 42: loc_embed shuffled - correct', 'counterfactual', 'loc_embed shuffled', '11446', 'loc_embed correct', '11438'),
    ('2B seed 43: loc_text shuffled - correct', 'counterfactual', 'loc_text shuffled', '11625', 'loc_text correct', '11624'),
    ('2B seed 43: loc_embed shuffled - correct', 'counterfactual', 'loc_embed shuffled', '11628', 'loc_embed correct', '11627'),
    ('4B seed 42: loc_text shuffled - correct', 'counterfactual', 'loc_text shuffled', '11447', 'loc_text correct', '11443'),
    ('4B seed 42: loc_embed shuffled - correct', 'counterfactual', 'loc_embed shuffled', '11448', 'loc_embed correct', '11444'),
    ('8B seed 42: loc_text shuffled - correct', 'counterfactual', 'loc_text shuffled', '11683', 'loc_text correct', '11682'),
    ('8B seed 42: loc_embed shuffled - correct', 'counterfactual', 'loc_embed shuffled', '11686', 'loc_embed correct', '11685'),
]

inventory = pd.DataFrame(
    [
        {
            'comparison': name,
            'kind': kind,
            'job_a': job_a,
            'job_b': job_b,
            'ready': all(
                (evaluation_root / job / 'predictions.jsonl').is_file()
                for job in (job_a, job_b)
            ),
        }
        for name, kind, _, job_a, _, job_b in comparison_specs
    ]
)
display(inventory)


In [ ]:
ready_names = set(inventory.loc[inventory['ready'], 'comparison'])
comparisons = [
    PredictionComparison(
        name=name,
        kind=kind,
        system_a=system_a,
        predictions_a=evaluation_root / job_a / 'predictions.jsonl',
        system_b=system_b,
        predictions_b=evaluation_root / job_b / 'predictions.jsonl',
    )
    for name, kind, system_a, job_a, system_b, job_b in comparison_specs
    if name in ready_names
]

intervals = analyze_comparisons(
    comparisons,
    metrics=PRIMARY_METRICS + DIRECT_GEOGRAPHY_METRICS,
    n_resamples=10_000,
    confidence_level=0.95,
    seed=42,
)
analysis_dir.mkdir(parents=True, exist_ok=True)
output_path = analysis_dir / 'paired_cluster_bootstrap_intervals.csv'
intervals.to_csv(output_path, index=False)
print(f'Wrote {output_path.relative_to(repo_root)}')


## Primary task-family metrics

Positive core differences favor the location-conditioned model. Negative counterfactual differences mean shuffled coordinates perform worse than correct coordinates.

In [ ]:
display(
    intervals[intervals['metric'].isin([metric.name for metric in PRIMARY_METRICS])][
        [
            'comparison',
            'metric',
            'score_a',
            'score_b',
            'difference_a_minus_b',
            'ci_low',
            'ci_high',
            'ci_excludes_zero',
        ]
    ].style.format(precision=4)
)


## Direct geography MCQ categories

In [ ]:
display(
    intervals[intervals['metric'].isin([metric.name for metric in DIRECT_GEOGRAPHY_METRICS])][
        [
            'comparison',
            'metric',
            'difference_a_minus_b',
            'ci_low',
            'ci_high',
            'ci_excludes_zero',
            'n_rows',
            'n_patches_with_metric',
        ]
    ].style.format(precision=4)
)
